In [8]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

# Добавляем папку src/ в пути импорта
ROOT = Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from data_prep import load_properties, merge_scenario_data, ScenarioDataset
from model import DeepSetsPredictor

from paths import TEST_PATH, PROPS_PATH, SAVE_DIR as WEIGHTS_DIR
print("✅ Импорт модулей завершён")

✅ Импорт модулей завершён


In [9]:
MODEL_PATH = WEIGHTS_DIR / "best_model.pt"
SCALERS_PATH = WEIGHTS_DIR / "scalers.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {DEVICE}")

🖥️ Using device: cuda


In [10]:
# Загружаем обученные скейлеры и список колонок
scalers = torch.load(SCALERS_PATH, map_location="cpu", weights_only=False)
prop_cols = scalers["prop_cols"]
n_props = len(prop_cols)

# Инициализируем и загружаем модель
model = DeepSetsPredictor(n_props=n_props).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False))
model.eval()

print(f"✅ Модель загружена. Размерность признаков: {n_props}")

✅ Модель загружена. Размерность признаков: 31


In [11]:
# 1. Загрузка скейлеров и списка признаков
scalers = torch.load(SCALERS_PATH, map_location='cpu', weights_only=False)
prop_cols = scalers['prop_cols']

# 2. Загрузка свойств и тестовых данных
props_df, _ = load_properties(PROPS_PATH)
test_df = pd.read_csv(TEST_PATH)
merged_test, valid_props = merge_scenario_data(test_df, props_df, prop_cols)

# 3. Инициализация датасета с сырыми данными (fit_scaler=False не трогает массивы)
test_ds = ScenarioDataset(merged_test, valid_props, fit_scaler=False)

# 4. Применяем обученные скейлеры к сырым данным
scaler_map = {
    'prop': scalers['prop_scaler'],
    'cond': scalers['cond_scaler'],
    'target': scalers['target_scaler']  # Для теста не используется, но нужен для сигнатуры
}
test_ds._transform(scaler_map)

# 5. DataLoader
test_dl = torch.utils.data.DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=0)
print(f"✅ Тестовый датасет готов: {len(test_ds)} сценариев | Признаков: {len(valid_props)}")

✅ Тестовый датасет готов: 40 сценариев | Признаков: 31


In [12]:
vis_preds, ox_preds = [], []

with torch.no_grad():
    for batch in test_dl:
        # Страховка от артефактов в тензорах
        props_t = torch.nan_to_num(batch['props'].to(DEVICE), nan=0.0, posinf=1.0, neginf=-1.0)
        
        v_pred, o_pred = model(
            props_t,
            batch['mask'].to(DEVICE),
            batch['conc'].to(DEVICE),
            batch['conditions'].to(DEVICE)
        )
        vis_preds.append(v_pred.cpu().numpy())
        ox_preds.append(o_pred.cpu().numpy())

vis_pred = np.concatenate(vis_preds)
ox_pred  = np.concatenate(ox_preds)
print("✅ Forward pass завершён")

✅ Forward pass завершён


In [13]:
# После сбора vis_pred и ox_pred (numpy arrays)
targets_inv = scalers['target_scaler'].inverse_transform(
    np.column_stack([vis_pred, ox_pred])
)

res = pd.DataFrame({
    'scenario_id': test_ds.scenario_ids,
    'Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %': targets_inv[:, 0],
    'Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm': targets_inv[:, 1]
})
res.to_csv("predictions.csv", index=False, encoding="utf-8")

In [14]:
test_ids = set(pd.read_csv(TEST_PATH)['scenario_id'].unique())
pred_ids = set(res['scenario_id'])

assert pred_ids == test_ids, "❌ Ошибка: состав scenario_id не совпадает с тестовым набором!"
assert not res.isnull().any().any(), "❌ Ошибка: найдены NaN в предсказаниях!"
assert res.duplicated('scenario_id').sum() == 0, "❌ Ошибка: найдены дубликаты scenario_id!"
assert len(res) == len(test_ids), "❌ Ошибка: количество строк не совпадает с количеством уникальных тестовых сценариев!"

print("🎉 Валидация пройдена успешно")

🎉 Валидация пройдена успешно
